# SageMaker + Ollama + ngrok (External Access)

Este notebook deja `OLLAMA_BASE_URL` publico por `ngrok` para consumo externo.

In [ ]:
# Celda 1: dependencias
!pip install -q pyngrok requests

In [ ]:
# Celda 2: verificar Ollama local en Docker
import subprocess

print(subprocess.getoutput("docker ps"))
print("\n---- OLLAMA LIST ----")
print(subprocess.getoutput("docker exec ollama ollama list"))
print("\n---- TAGS (localhost) ----")
print(subprocess.getoutput("curl -s http://127.0.0.1:11434/api/tags"))

In [ ]:
# Celda 3: (opcional) descargar modelo
# Cambia el modelo si quieres otro
!docker exec ollama ollama pull qwen2.5:7b-instruct
!docker exec ollama ollama list

In [ ]:
# Celda 4: abrir tunnel ngrok al puerto 11434
import os
import json
import time
import pathlib
import requests
from pyngrok import ngrok

# OPCION A (recomendada): exporta la variable antes de correr el notebook:
# export NGROK_AUTHTOKEN="<TU_TOKEN>"
token = os.environ.get("NGROK_AUTHTOKEN", "").strip()

# OPCION B: descomenta y pega el token manualmente (no recomendado para guardar en notebook)
# token = "<TU_TOKEN_NGROK>"

if not token:
    raise RuntimeError("Falta NGROK_AUTHTOKEN. Define la variable de entorno antes de ejecutar.")

ngrok.set_auth_token(token)

# Cerrar tuneles previos para evitar conflicto
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public = ngrok.connect(addr=11434, proto="http")
public_url = public.public_url.rstrip("/")
print("OLLAMA_NGROK_URL:", public_url)

# Guardar endpoint para otros scripts
repo = pathlib.Path.cwd()
out_dir = repo / "data" / "runtime"
out_dir.mkdir(parents=True, exist_ok=True)
payload = {
    "ollama_local_url": "http://127.0.0.1:11434",
    "ollama_ngrok_url": public_url,
    "ollama_base_url": public_url,
    "created_at_epoch": int(time.time())
}
(out_dir / "ollama_endpoint.json").write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado:", out_dir / "ollama_endpoint.json")

In [ ]:
# Celda 5: prueba externa via ngrok (tags + generate)
import json
import pathlib
import requests

repo = pathlib.Path.cwd()
cfg = json.loads((repo / "data" / "runtime" / "ollama_endpoint.json").read_text(encoding="utf-8"))
base = cfg["ollama_base_url"].rstrip("/")

print("BASE:", base)
print("\nTAGS:")
print(requests.get(f"{base}/api/tags", timeout=30).json())

prompt = "Responde breve: confirma que el endpoint de Ollama por ngrok funciona."
body = {
    "model": "qwen2.5:7b-instruct",
    "prompt": prompt,
    "stream": False
}
r = requests.post(f"{base}/api/generate", json=body, timeout=120)
r.raise_for_status()
print("\nGENERATE:")
print(r.json().get("response", ""))